<a href="https://colab.research.google.com/github/DhiSys-AI/MRM-Trace/blob/main/notebooks/001%20-%20Getting%20Started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mrm-trace · 001: Getting Started

**mrm-trace** is scientific instrumentation for Managed-Retention Memory (MRM) research.  
It collects, parses, labels, and analyses LLM inference memory access traces.

**Primary metrics:** retention duration · write-once ratio · read frequency · working set size

---

### What this notebook covers

| Section | What you'll learn |
|---|---|
| 1. Install & verify | Install from PyPI, confirm version |
| 2. Data model | Three-layer architecture, full 18-column trace schema |
| 3. Build a synthetic trace | Create a realistic trace DataFrame |
| 4. Label the trace | Classify addresses into weight / kv_cache regions |
| 5. Analyse | Compute all MRM metrics in one call |
| 6. Interpret results | Retention, write-once, locality, IAI, suitability labels |
| 7. Export | Write a publication-ready run directory |
| 8. Schema versioning | Validate Parquet files on load |
| 9. Run validity | Classify clean / marginal / contaminated runs |
| Appendix | Run individual analysers |

> **Colab note:** `perf mem` collection requires Linux PMU + root (`sudo`).  
> This notebook uses the Python API with synthetic data — everything runs without root or model files.

## 1. Install & Verify

In [ ]:
# Install mrm-trace and the optional plotting extras
!pip install -q "mrm-trace[plots]"

In [ ]:
import mrm_trace
print(f"mrm-trace version: {mrm_trace.__version__}")

# Confirm key dependencies
import pandas as pd
import numpy as np
import pyarrow
print(f"pandas {pd.__version__} | numpy {np.__version__} | pyarrow {pyarrow.__version__}")

## 2. The Three-Layer Data Model

Every mrm-trace run produces three layers of output:

```
Layer 1: OBSERVED DATA     → trace.parquet, region_map.parquet, kv_block_lifecycle.parquet
Layer 2: OBSERVER EFFECT   → metadata.json / observer_effect section
Layer 3: MACHINE CONTEXT   → metadata.json / hardware + software + baseline + thermal
```

Layers 2 and 3 allow trace values (latency, retention) to be **independently audited and compared** across machines and papers.

### Canonical trace.parquet schema (18 columns)

| Column | Type | Description |
|---|---|---|
| `run_id` | str | Unique run identifier |
| `sample_id` | int | Sequential sample index |
| `timestamp_ns` | int | Absolute timestamp in nanoseconds |
| `phase` | str | `weight_load` / `generation` / `teardown` |
| `address` | int | Raw memory address |
| `address_page` | int | Page-aligned address (`address >> 12`) |
| `op_type` | str | `load` or `store` |
| `symbol` | str | Resolved symbol name |
| `dso` | str | Shared library / DSO |
| `region_type` | str | `weight` / `kv_cache` / `activation` / `unknown` |
| `region_id` | int | Stable region identifier |
| `access_latency_ns` | int | Memory access latency from `perf mem` |
| `cache_level` | str | `L1` / `L2` / `L3` / `DRAM` |
| `numa_node` | int | NUMA node of the access |
| `cpu_id` | int | CPU core ID |
| `thread_id` | int | OS thread ID |
| `token_index` | int | Index of the current token being generated |
| `context_tokens` | int | Context length at time of access |

> **Phase tagging is essential (ADR-3):** `weight_load` / `generation` / `teardown` phases  
> distinguish weight accesses from KV cache accesses, which are indistinguishable by address alone.  
> Our synthetic trace only uses the minimal required columns — a real `perf mem` run populates all 18.

### Output directory layout

```
results/<model_id>/<run_id>/
├── trace.parquet                  ← labelled memory access trace
├── region_map.parquet             ← one row per region (weight, kv_cache, …)
├── kv_block_lifecycle.parquet     ← per-block write / read / eviction timestamps
├── metrics.csv                    ← per-region-type summary (human-readable)
├── metadata.json                  ← hardware, software, observer effect, run validity
├── manifest.json                  ← SHA-256 checksums for all files
└── raw/
    ├── perf.data
    ├── perf_script.txt
    └── memray.bin
```

### MRM suitability labels

| Label | Criteria | Typical region |
|---|---|---|
| `high_mrm` | write-once ≥ 0.8 **and** retention p99 ≥ 10 s | Model weights |
| `medium_mrm` | write-once ≥ 0.5 **and** retention p50 ≥ 1 s | Activations |
| `low_mrm` | everything else | Short-lived KV blocks |

## 3. Build a Synthetic Trace

We create a realistic trace DataFrame with:
- A **weight region** - one store, many reads, 30 s retention window → `high_mrm`
- A **KV-cache region** - many blocks each written once and read a few times → `low_mrm`

The canonical trace schema requires these columns:
`run_id, sample_id, timestamp_ns, address, address_page, op_type, region_type, region_id, phase, symbol, dso`

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

rows = []
sample_id = 0

# ── Weight region ────────────────────────────────────────────────────────────
# One store at t=0, then 500 reads spread over 30 seconds.
# This simulates model weights being loaded once and repeatedly read during inference.
WEIGHT_ADDR = 0x7F0000000000
N_WEIGHT_READS = 500
WEIGHT_RETENTION_S = 30.0

rows.append(dict(
    run_id="demo_run", sample_id=sample_id,
    timestamp_ns=0,
    address=WEIGHT_ADDR, address_page=WEIGHT_ADDR >> 12,
    op_type="store", region_type="weight", region_id=1,
    phase="weight_load", symbol="ggml_init", dso="libggml.so",
))
sample_id += 1

for i in range(N_WEIGHT_READS):
    t_ns = int((i + 1) * (WEIGHT_RETENTION_S / N_WEIGHT_READS) * 1e9)
    addr = WEIGHT_ADDR + int(rng.integers(0, 4096)) * 64
    rows.append(dict(
        run_id="demo_run", sample_id=sample_id,
        timestamp_ns=t_ns,
        address=addr, address_page=addr >> 12,
        op_type="load", region_type="weight", region_id=1,
        phase="generation", symbol="ggml_vec_dot_q4_K", dso="libggml.so",
    ))
    sample_id += 1

# ── KV-cache region ───────────────────────────────────────────────────────────
# 50 KV blocks each written once, then read 5 times within a short window.
# KV blocks are 256 bytes each (sub-page granularity).
KV_ADDR = 0x7F1000000000
N_KV_BLOCKS = 50
N_READS_PER_BLOCK = 5
KV_WINDOW_S = 2.0

for i in range(N_KV_BLOCKS):
    t_write_ns = int((1.0 + i * (KV_WINDOW_S / N_KV_BLOCKS)) * 1e9)
    addr = KV_ADDR + i * 256
    rid = 2 + i
    rows.append(dict(
        run_id="demo_run", sample_id=sample_id,
        timestamp_ns=t_write_ns,
        address=addr, address_page=addr >> 12,
        op_type="store", region_type="kv_cache", region_id=rid,
        phase="generation", symbol="llama_kv_cache_update", dso="libllama.so",
    ))
    sample_id += 1
    for j in range(N_READS_PER_BLOCK):
        rows.append(dict(
            run_id="demo_run", sample_id=sample_id,
            timestamp_ns=t_write_ns + int((j + 1) * 0.01 * 1e9),
            address=addr, address_page=addr >> 12,
            op_type="load", region_type="kv_cache", region_id=rid,
            phase="generation", symbol="llama_kv_cache_update", dso="libllama.so",
        ))
        sample_id += 1

trace = pd.DataFrame(rows)
print(f"Trace shape: {trace.shape}")
print(f"\nRegion types: {trace['region_type'].value_counts().to_dict()}")
print(f"Op types:     {trace['op_type'].value_counts().to_dict()}")
trace.head()

## 4. Label the Trace

`TraceLabeller` is a **streaming classifier** - it processes rows as a generator and never loads the full trace into RAM (ADR-2).  
It classifies each row into a `region_type` and assigns a stable `region_id`, then produces:
- `region_map` - one row per region with page bounds and access counts
- `kv_lifecycle` - per-block write / read / eviction timestamps

> Because our trace already has `region_type` set (pre-labelled synthetic data), the labeller passes those rows through unchanged. In a real run the labeller classifies by symbol name and DSO.

In [ ]:
from mrm_trace.labeller import TraceLabeller

labeller = TraceLabeller()

# label() is a generator - consume it fully before calling region_map() / kv_lifecycle()
labelled_rows = list(labeller.label(trace.to_dict(orient="records")))
labelled_trace = pd.DataFrame(labelled_rows)

region_map = pd.DataFrame(labeller.region_map())
kv_lifecycle = pd.DataFrame(labeller.kv_lifecycle())

print(f"Labelled trace:  {labelled_trace.shape}")
print(f"Region map:      {region_map.shape}  (one row per region)")
print(f"KV lifecycle:    {kv_lifecycle.shape}  (one row per KV block)")

In [ ]:
print("Region map:")
region_map[["region_id", "region_type", "access_count", "first_seen_ns", "last_seen_ns"]].head(10)

In [ ]:
if not kv_lifecycle.empty:
    print("KV block lifecycle (first 5 blocks):")
    display(kv_lifecycle.head())
else:
    print("No KV lifecycle data (no kv_cache region in trace)")

## 5. Analyse

`compute_all()` runs all six analysers in one call and returns a dict with these keys:

| Key | Description |
|---|---|
| `retention_per_region` | First-write → last-read window per region |
| `retention_summary` | p50 / p90 / p99 / mean per region_type |
| `write_once` | Fraction of written addresses written exactly once per region |
| `read_freq` | Total reads / writes / read fraction per region |
| `working_set_per_region` | Unique pages and bytes per region |
| `working_set_summary` | Aggregated working set per region_type |
| `locality_per_region` | Stride distribution and same-page fraction per region |
| `locality_summary` | Aggregated locality per region_type |
| `iai` | Inter-access interval p50 / p90 / p99 / mean per region_type |
| `suitability` | MRM suitability label per region_type |

In [ ]:
from mrm_trace.analyser import compute_all

results = compute_all(labelled_trace)
print("Result keys:", list(results.keys()))

## 6. Interpret Results

In [ ]:
print("=== Retention summary (per region_type) ===")
display(results["retention_summary"])

print("\n=== Retention per region ===")
display(results["retention_per_region"][["region_id", "region_type", "retention_s", "first_write_ns", "last_read_ns"]])

In [ ]:
print("=== Write-once ratio (per region) ===")
print("  1.0 = every written address was written exactly once (ideal for MRM)")
display(results["write_once"][["region_id", "region_type", "write_once_ratio", "n_written_addresses", "n_write_once_addresses"]])

In [ ]:
print("=== Read frequency (per region) ===")
display(results["read_freq"][["region_id", "region_type", "total_reads", "total_writes", "read_fraction", "reads_per_write"]])

In [ ]:
print("=== Working set per region ===")
display(results["working_set_per_region"][["region_id", "region_type", "unique_pages", "working_set_bytes"]])

print("\n=== Working set summary (per region_type) ===")
display(results["working_set_summary"])

In [ ]:
print("=== Locality per region ===")
print("  mean_stride_bytes: average distance between consecutive accesses")
print("  same_page_fraction: fraction of consecutive accesses that stay on the same 4K page")
display(results["locality_per_region"][["region_id", "region_type", "n_accesses", "mean_stride_bytes", "median_stride_bytes", "same_page_fraction"]])

print("\n=== Locality summary (per region_type) ===")
display(results["locality_summary"])

In [ ]:
print("=== Inter-access interval (IAI) - per region_type ===")
print("  IAI measures the time between consecutive accesses to the same address.")
print("  Units: nanoseconds (ns)  |  divide by 1_000 for µs, by 1_000_000 for ms")
print("  Low IAI → address is accessed in rapid bursts (high temporal locality).")
print("  High IAI → long gaps between accesses (weight-like access pattern).")
print()
display(results["iai"])

In [ ]:
print("=== MRM Suitability labels ===")
print("  high_mrm  : write_once ≥ 0.8  AND  retention_p99 ≥ 10 s  (e.g. model weights)")
print("  medium_mrm: write_once ≥ 0.5  AND  retention_p50 ≥ 1 s")
print("  low_mrm   : everything else   (e.g. short-lived KV blocks)")
print()
print("  The 'suitability' column is a categorical label (string), not a numeric score.")
print("  Expected results for this synthetic trace:")
print("    weight   → high_mrm  (write-once ratio ≈ 1.0, retention ≈ 30 s)")
print("    kv_cache → low_mrm   (many short-lived blocks, retention < 2 s)")
print()
display(results["suitability"])

## 7. Export a Publication-Ready Run Directory

`RunExporter` writes all output files to `results/<run_id>/` and produces a `manifest.json` with SHA-256 checksums for every file.

In [ ]:
from mrm_trace.reporter import RunExporter

RUN_DIR = "/content/demo_run_001"

exporter = RunExporter(RUN_DIR)
export_result = exporter.export(
    trace=labelled_trace,
    region_map=region_map,
    kv_lifecycle=kv_lifecycle,
    analysis=results,
    metadata={"run_id": "demo_run_001", "model": "synthetic"},
    run_id="demo_run_001",
    figures=True,   # set False if matplotlib is not installed
)

print("Export complete. Files written:")
for path in sorted(export_result.written_files):
    print(f"  {path}")

In [ ]:
import os, json

# Show the directory tree
for root, dirs, files in os.walk(RUN_DIR):
    level = root.replace(RUN_DIR, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        print(f"{indent}    {f}  ({size:,} bytes)")

In [ ]:
# Read back the metrics CSV - human-readable summary
metrics_path = os.path.join(RUN_DIR, "metrics.csv")
if os.path.exists(metrics_path):
    print("metrics.csv:")
    display(pd.read_csv(metrics_path))

In [ ]:
# Inspect the manifest (SHA-256 checksums for reproducibility)
manifest_path = os.path.join(RUN_DIR, "manifest.json")
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)
    print("manifest.json:")
    for entry in manifest.get("files", []):
        print(f"  {entry['path']:45s}  {entry['sha256'][:16]}...")

## 8. Schema Versioning

All Parquet output files carry a `mrm_trace_schema_version` in their metadata.  
Readers validate major-version compatibility on load - a major bump is a breaking change.

In [ ]:
from mrm_trace.schema_version import check_parquet_schema, CURRENT_VERSIONS

print("Current schema versions:")
for name, version in CURRENT_VERSIONS.items():
    print(f"  {name:25s}  v{version}")

# Validate the trace we just exported
trace_path = os.path.join(RUN_DIR, "trace.parquet")
if os.path.exists(trace_path):
    try:
        check_parquet_schema(trace_path, "trace")
        print(f"\ntrace.parquet schema: OK")
    except Exception as e:
        print(f"\ntrace.parquet schema error: {e}")

## 9. Run Validity Classification

Every mrm-trace run is automatically classified based on the tracer's own overhead.  
This is Layer 2 of the data model — it lets readers judge whether the measurement itself disturbed the system.

| Class | CPU overhead | Mem fraction | Throttle events | Baseline CPU |
|---|---|---|---|---|
| `clean` | < 10 % | < 5 % of target RSS | 0 | < 15 % |
| `marginal` | < 20 % | < 15 % of target RSS | ≤ 2 | — |
| `contaminated` | ≥ 20 % | ≥ 15 % of target RSS | > 2 | — |

Contaminated runs are archived but **excluded from aggregated metrics and paper figures**.

In [ ]:
from mrm_trace.telemetry.validity import classify_run

# Simulate three different observer-overhead scenarios
scenarios = [
    dict(name="clean run",
         observer_cpu_overhead_pct_max=5.0, observer_memory_frac=0.02,
         thermal_throttle_events=0, baseline_cpu_pct=8.0),
    dict(name="marginal run",
         observer_cpu_overhead_pct_max=15.0, observer_memory_frac=0.08,
         thermal_throttle_events=1, baseline_cpu_pct=12.0),
    dict(name="contaminated run",
         observer_cpu_overhead_pct_max=25.0, observer_memory_frac=0.20,
         thermal_throttle_events=5, baseline_cpu_pct=18.0),
]

print(f"{'Scenario':<20}  {'Validity class':<15}  Details")
print("-" * 70)
for s in scenarios:
    result = classify_run(
        observer_cpu_overhead_pct_max=s["observer_cpu_overhead_pct_max"],
        observer_memory_frac=s["observer_memory_frac"],
        thermal_throttle_events=s["thermal_throttle_events"],
        baseline_cpu_pct=s["baseline_cpu_pct"],
    )
    print(f"{s['name']:<20}  {result.validity_class:<15}  "
          f"cpu={s['observer_cpu_overhead_pct_max']:.0f}%  "
          f"mem_frac={s['observer_memory_frac']:.0%}  "
          f"throttles={s['thermal_throttle_events']}")

print()
print("In a real run, validity_class is written to metadata.json and")
print("checked by RunExporter before including the run in aggregated metrics.")

## Appendix: Running Individual Analysers

Each analyser can be imported and called independently if you only need one metric.

In [ ]:
from mrm_trace.analyser import (
    compute_retention,
    compute_write_once,
    compute_read_freq,
    compute_working_set,
    compute_locality,
    compute_iai,
    classify_suitability,
)

# Retention only
retention_per_region, retention_summary = compute_retention(labelled_trace)
print("Retention p99 per region_type (seconds):")
print(retention_summary[["region_type", "retention_p99_s"]].to_string(index=False))

# Write-once only
write_once = compute_write_once(labelled_trace)
print("\nWrite-once ratio per region_type:")
print(write_once.groupby("region_type")["write_once_ratio"].mean().to_string())

## Next Steps

### CLI commands

| Command | What it does | Status |
|---|---|---|
| `mrm-trace validate --config <yaml>` | Validate YAML config and environment | ✅ Available |
| `mrm-trace plan --config <yaml>` | Preview run matrix (model × context × batch) | ✅ Available |
| `mrm-trace run --config <yaml>` | Collect raw traces (Linux + sudo required) | ✅ Available |
| `mrm-trace analyse --input <dir>` | Parse, label, compute metrics from raw artifacts | 🔜 Coming soon |
| `mrm-trace report --input <dir>` | Generate aggregated CSVs and figures | 🔜 Coming soon |
| `mrm-trace export --input <dir>` | Package outputs for HuggingFace / Zenodo release | 🔜 Coming soon |

### Python API

```python
from mrm_trace.api import Experiment

# Load from YAML
exp = Experiment.from_yaml("config/default_experiment.yaml")

# Validate config and environment
exp.validate()

# Preview the run matrix
runs = exp.plan()
for r in runs:
    print(r.run_id, r.model.id, r.context_length)
```

### Resources

| Resource | Link |
|---|---|
| Source code | [github.com/DhiSys-AI/MRM-Trace](https://github.com/DhiSys-AI/MRM-Trace) |
| Bug reports | [github.com/DhiSys-AI/MRM-Trace/issues](https://github.com/DhiSys-AI/MRM-Trace/issues) |
| Email | info@dhisys.co.uk |

---
*mrm-trace v0.1.2 · DhiSys AI*